# Baseline Model

As the goal is to classify whether or not a patient will be readmitted, it boils down to a binary classification problem with multiple independent variables as the features (EHRs). Using Logistic Regression as the baseline model is fast to train and can serve as the floor for future models.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

train_df = pd.read_csv('../data/train.csv')
train_df['gender'] = train_df['gender'].map({'Male': 0, 'Female': 1})

X = train_df.drop(columns=['readmitted_30_days', 'patient_id'])

y = train_df['readmitted_30_days']


In [2]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

X_train.shape

(24000, 13)

In [3]:
X_val.shape

(6000, 13)

In [4]:
y_train.value_counts(normalize=True)

readmitted_30_days
0    0.802583
1    0.197417
Name: proportion, dtype: float64

In [5]:
y_val.value_counts(normalize=True)

readmitted_30_days
0    0.8025
1    0.1975
Name: proportion, dtype: float64

## Train/Val Split - Findings
- 80/20 split
- Train: ~80.26% / ~19.74%
- Val: ~80.25% / ~19.75%
- Balance preserved as expected
- Omitted patient id
- Mapped gender to numeric values for training


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

model = LogisticRegression(random_state=SEED).fit(X_train_scaled, y_train)

y_preds = model.predict_proba(X_val_scaled)

y_probs = [row[1] for row in y_preds]

auc_score = roc_auc_score(y_val, y_probs)

auc_score


0.48655090675675083

In [19]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='stratified', random_state=SEED).fit(X_train, y_train)

dummy_preds = dummy.predict_proba(X_val)
dummy_probs = [row[1] for row in dummy_preds]

dummy_auc = roc_auc_score(y_val, dummy_probs)
dummy_auc

0.4964036436768012

## Baseline Model - Findings
- Model: Logistic Regression (scaled features)
- AUC on validation set: 0.4865
- Compared against DummyClassifer: AUC 0.4964
- Logistic Regression performs roughly the same as random guessing
- Conclusion: little to no linear relationship between current features and readmission value
- Next step: Try a tree-based model (boosted decision tree) to test for non-linear patterns logistic regression can't capture
